# Lab - Tool Calls: Contract, Dispatch, Matched Results

Three tasks that follow one tool call from start to finish - from the developer's side: the contract Claude reads, the dispatcher that turns a request into a real operation, and the result that goes back to the right block.

| Task | What you learn |
| --- | --- |
| 1 | A tight tool contract: required field, ID pattern, no extra fields |
| 2 | dispatch(): unknown tools and bad inputs become error results, never crashes |
| 3 | One result per tool_use block, matched by id, then Claude answers |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import re

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery. "
    "Answer in one or two sentences."
)

ORDERS = {
    "ORD-12345678": {"item": "jacket", "status": "delivered", "delivered_days_ago": 20},
}


def lookup_order(order_id):   # the function behind the tool: plain Python
    return ORDERS.get(order_id, {"found": False, "message": "No order with this ID."})


QUESTION = [{"role": "user",
             "content": "Can I still return the jacket from order ORD-12345678?"}]

---

## Task 1 - A tool contract Claude can read

In [ ]:
# ============================================================
# TASK 1 - A tool contract Claude can read
# ============================================================
#
# WHAT TO DO
#   Finish the definition of lookup_order: an input schema that
#   requires one order_id in the store's exact format and allows
#   nothing else. Send it with the question and look at what
#   comes back.
#
# WHY IT MATTERS
#   A tool definition is a description, not code - Claude reads
#   it like documentation. The tighter the contract, the fewer
#   bad calls reach your code: a pattern for the ID, the
#   required field, and no extra fields. Claude does not answer;
#   it stops with stop_reason "tool_use" and asks for the tool.
#   Nothing has run yet.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The tool definition is a description, not code".
#
# HOW TO DO IT
#   Three blanks:
#
#       r"^ORD-\d{8}$"   ORD- and exactly eight digits
#       False            no fields beyond the ones listed
#       TOOLS            the tools list, sent with the request
# ============================================================

# TODO: replace each ... below with the value named beside it
TOOLS = [{
    "name": "lookup_order",
    "description": "Look up an order by its ID. Returns the item, the status and "
                   "how many days ago it was delivered.",
    "input_schema": {
        "type": "object",
        "properties": {
            "order_id": {"type": "string", "pattern": ...,   # r"^ORD-\d{8}$"
                         "description": "The order ID, like ORD-12345678"},
        },
        "required": ["order_id"],
        "additionalProperties": ...,   # False - nothing beyond the listed fields
    },
}]

response = client.messages.create(
    model=MODEL, max_tokens=300, output_config={"effort": "low"},
    system=SYSTEM, tools=..., messages=QUESTION)   # TOOLS

print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "tool_use":
        print("Claude asks for:", block.name, block.input, "| id", block.id)

check("tool_contract", tools=TOOLS, response=response)

---

## Task 2 - Dispatch by name, check the input

In [ ]:
# ============================================================
# TASK 2 - Dispatch by name, check the input
# ============================================================
#
# WHAT TO DO
#   Finish dispatch(): it takes one tool_use block and always
#   returns one tool_result for it. An unknown tool name, or an
#   order ID that breaks the contract, comes back as an error
#   result; a valid call runs the function.
#
# WHY IT MATTERS
#   Claude only asks - your code runs the tool. That code is the
#   boundary: it decides which function a name maps to, and it
#   checks the input before anything touches your data. A
#   dispatcher that raises on a bad call kills the conversation;
#   one that answers with an error result lets Claude explain or
#   try again. And every result carries the id of the block that
#   asked for it.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now our code does its part".
#
# HOW TO DO IT
#   Three blanks:
#
#       block.id        the id the result must carry
#       HANDLERS        the names your code knows
#       r"ORD-\d{8}"    the contract's order ID format
# ============================================================

HANDLERS = {"lookup_order": lookup_order}   # provided: tool name -> function

# TODO: replace each ... below with the value named beside it
def dispatch(block):
    def result(payload, is_error=False):
        return {"type": "tool_result", "tool_use_id": ...,   # block.id
                "content": json.dumps(payload), "is_error": is_error}

    if block.name not in ...:   # HANDLERS
        return result({"error": f"unknown tool {block.name!r}"}, is_error=True)
    order_id = str(block.input.get("order_id", ""))
    if not re.fullmatch(..., order_id):   # r"ORD-\d{8}"
        return result({"error": f"order_id {order_id!r} is not like ORD-12345678"},
                      is_error=True)
    return result(HANDLERS[block.name](**block.input))

try:
    for block in response.content:
        if block.type == "tool_use":
            print(dispatch(block))
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("dispatch", dispatch=dispatch)

---

## Task 3 - Send back the matched result

In [ ]:
# ============================================================
# TASK 3 - Send back the matched result
# ============================================================
#
# WHAT TO DO
#   Build the conversation for the second request: the question,
#   Claude's reply exactly as it came, and one user turn with a
#   result for every tool_use block. Send it with the same tools
#   and print the answer.
#
# WHY IT MATTERS
#   A tool call is a round trip: Claude asks with tool_use, your
#   code answers with tool_result, and only then does Claude
#   answer the customer. The results go back in one user turn,
#   matched to the blocks by id - one result per block, none
#   missing. Claude now has the order in front of it.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Then two messages go into the conversation".
#
# HOW TO DO IT
#   Three blanks:
#
#       response.content   Claude's reply, exactly as it came
#       block              each tool_use block goes to dispatch()
#       messages           the whole conversation, sent again
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    messages = QUESTION + [
        {"role": "assistant", "content": ...},   # response.content
        {"role": "user", "content": [dispatch(...) for block in response.content   # block
                                     if block.type == "tool_use"]},
    ]
    final = client.messages.create(
        model=MODEL, max_tokens=300, output_config={"effort": "low"},
        system=SYSTEM, tools=TOOLS, messages=...)   # messages
except (TypeError, AttributeError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("stop_reason:", final.stop_reason)
print(final.content[0].text)

check("matched_result", messages=messages, final=final)

---

## Done

A tool call is a round trip, and your code owns the middle: a contract tight enough that bad calls are rare, a dispatcher that turns every request into exactly one result - errors included - and results matched to their blocks by id.